# tcpyVPI: ERA5 vPI and GPIv, installed **from GitHub**

This notebook is the same workflow as the other ERA5 examples, but it installs
`tcpyVPI` **directly from the GitHub repository** rather than from PyPI. Use this
when you want the current state of `main` (or a specific tag) rather than waiting
for a PyPI release.

IMPORTANT: this notebook reads ERA5 data remotely via the NCAR THREDDS server.
Sometimes THREDDS throws a `NetCDF: DAP server error` when reading the data -- it
seems to be somewhat random whether/when it happens and is not due to this
notebook. If it happens, run it again.

**Cite this package:**  
Chavas, D. Sanchez, J. O., and A. Kruskie (2026). *tcpyVPI*. https://doi.org/10.5281/zenodo.19319996

[![PyPI version](https://img.shields.io/pypi/v/tcpyVPI.svg)](https://pypi.org/project/tcpyVPI/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19319996.svg)](https://doi.org/10.5281/zenodo.19319996)

**Author:** Dan Chavas (2026)


## 0. Install from GitHub

`pip` installs straight from the repo with the `git+https://` scheme. Pin a tag
for reproducible results, or track `main` for the newest code.


In [ ]:
# Pinned to a release tag (reproducible) -- change the tag to move versions
!pip install -q "git+https://github.com/drchavas/tcpyVPI.git@v1.1.0"

# Or track the latest commit on main instead:
# !pip install -q "git+https://github.com/drchavas/tcpyVPI.git@main"

# Force a reinstall if you already have tcpyVPI from PyPI in this runtime:
# !pip install -q --force-reinstall --no-deps "git+https://github.com/drchavas/tcpyVPI.git@main"

!pip install -q tcpyPI cartopy

### Confirm it really came from GitHub, not PyPI

A PyPI install shows a bare version (`tcpyVPI==1.1.0`); a git install shows the
repository URL and the exact commit it was built from.


In [ ]:
import subprocess, tcpyVPI

print("version :", tcpyVPI.__version__)
print("path    :", tcpyVPI.__file__)
print()
freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
line = [l for l in freeze.splitlines() if l.lower().startswith("tcpyvpi")]
print("pip freeze:", line[0] if line else "(not found)")
print()
if line and "git+" in line[0]:
    print("OK - installed from GitHub")
else:
    print("WARNING - this looks like the PyPI build, not the GitHub one.")
    print("Re-run the install cell with --force-reinstall.")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from tcpyVPI import (
    run_vpigpiv,
    load_era5_data,
    compute_gpiv_from_dataset,
)

## 1. One-call version

`run_vpigpiv` loads the ERA5 fields, computes everything, and draws the standard
diagnostic panels. This is the quickest way to check the install works.


In [ ]:
year, month = 2022, 9

results = run_vpigpiv(year, month, data_source='monthly', plot=True)
print("\nVariables computed:", list(results.data_vars))

## 2. Load and compute separately, then make your own plot

Splitting the two steps lets you keep the raw ERA5 fields around and plot
whatever you like.


In [ ]:
ds = load_era5_data(year, month, data_source='monthly')
results = compute_gpiv_from_dataset(ds)

results

### Sample plot: potential intensity vs. ventilated potential intensity

The difference between the two panels is the whole point of the ventilated PI.
`PI` is the thermodynamic ceiling. `vPI` is what the storm can actually reach
once shear and mid-level dry air are accounted for, and it cuts off sharply to
zero where the ventilation index exceeds 0.145.


In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)
fig, axes = plt.subplots(3, 1, figsize=(11, 12),
                         subplot_kw={'projection': proj},
                         constrained_layout=True)

panels = [
    ('PI',                results['PI'],                'Potential intensity  [m s$^{-1}$]',        dict(vmin=0, vmax=100, cmap='viridis')),
    ('vPI',               results['vPI'],               'Ventilated PI  [m s$^{-1}$]',              dict(vmin=0, vmax=100, cmap='viridis')),
    ('ventilation_index', results['ventilation_index'], 'Ventilation index  [-]',                   dict(vmin=0, vmax=0.3, cmap='plasma')),
]

for ax, (name, field, label, kw) in zip(axes, panels):
    p = field.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                              add_colorbar=True, cbar_kwargs={'label': label,
                                                              'shrink': 0.85},
                              **kw)
    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=2)
    ax.set_extent([-180, 180, -50, 50], crs=ccrs.PlateCarree())
    ax.set_title(f"{name}   ERA5 {year}-{month:02d}")

plt.show()

### The reduction from ventilation

`PI - vPI` is how much intensity the environment takes away. It is largest where
shear is strong and the mid-troposphere is dry -- the subtropics, and the eastern
sides of the basins.


In [ ]:
reduction = (results['PI'] - results['vPI']).where(results['PI'] > 0)

fig, ax = plt.subplots(figsize=(11, 4.5),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)},
                       constrained_layout=True)
reduction.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                          vmin=0, vmax=100, cmap='magma_r',
                          cbar_kwargs={'label': 'PI $-$ vPI  [m s$^{-1}$]', 'shrink': 0.85})
ax.coastlines(linewidth=0.6)
ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=2)
ax.set_extent([-180, 180, -50, 50], crs=ccrs.PlateCarree())
ax.set_title(f"Intensity lost to ventilation   ERA5 {year}-{month:02d}")
plt.show()

## 3. Sanity check on the values

Quick check that the numbers are physically sensible. Tropical PI in September
should peak somewhere around 80-95 m/s over the warm pool, not above ~110.

Versions of tcpyVPI **up to and including v1.0.1** passed surface pressure to
`tcpyPI` in Pa instead of hPa, and specific humidity where mixing ratio was
expected, which biased PI high by roughly 25%. Both were fixed in v1.1.0. If you
see PI maxima well above 110 m/s here, check which version you installed.


In [ ]:
for name in ['PI', 'vPI', 'ventilation_index', 'Chi', 'VWS']:
    f = results[name]
    print(f"{name:18s} min={float(f.min()):8.3f}   "
          f"mean={float(f.mean()):8.3f}   max={float(f.max()):8.3f}")

pi_max = float(results['PI'].max())
print()
if pi_max > 110:
    print(f"WARNING: PI max = {pi_max:.1f} m/s looks too high - are you on a pre-v1.1.0 build?")
else:
    print(f"PI max = {pi_max:.1f} m/s - in the expected range.")